# 22. Gradient boosting
Boosting adds models sequentially to improve an objective. In gradient boosting, each new learner targets directions that reduce the current loss. Learning rate controls how strongly each learner contributes. Histogram-based estimators group feature values into bins to make training more efficient. Early stopping uses validation evidence to stop adding iterations.

Balance learning rate with the iteration budget. Compare validation curves rather than assuming more iterations help. Keep the early-stopping validation data separate from the final test set. When cross-validating, internal validation must remain within the current training fold. Missing-value and categorical support depend on the estimator and representation.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X, y = load_breast_cancer(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
candidate = HistGradientBoostingClassifier(max_iter=40, learning_rate=0.1, max_leaf_nodes=15, early_stopping=True, random_state=42) # Define bounded boosting.
cv = StratifiedKFold(3, shuffle=True, random_state=42) # Evaluate on development data.
scores = cross_val_score(candidate, X_dev, y_dev, cv=cv, scoring="accuracy") # Internal early stopping sees training-fold data only.
print("Fold accuracies:", scores.round(3)) # Inspect spread before choosing.
print("Mean:", round(scores.mean(), 3)) # Summarize evaluation.
assert len(scores) == 3 # Verify fold count.

## Native categorical histogram boosting
HistGradientBoosting can recognize pandas categorical columns with categorical_features="from_dtype". A category column describes a vocabulary, not a ranked number. Convert relevant columns to dtype category; nullable numeric Float64 remains numeric, not automatically categorical. Cardinality must respect the estimator’s bin limits. Unknown categories are treated as missing by the categorical prediction path. Native handling avoids a one-hot expansion but does not remove the need for a correct split, feature availability checks, or schema validation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(42) # Reproducible example rows.
X = pd.DataFrame({"city": pd.Categorical(np.tile(["A", "B", "C"], 60)), "value": pd.array(rng.normal(size=180), dtype="Float64")}) # Explicit category and nullable numeric columns.
y = ((X["city"] == "A") | (X["value"] > 0.5)).astype(int) # Construct an educational target.
Xt, Xv, yt, yv = train_test_split(X, y, stratify=y, random_state=42) # Development validation split.
model = HistGradientBoostingClassifier(categorical_features="from_dtype", max_iter=30, min_samples_leaf=5, random_state=42).fit(Xt, yt) # Learn native category splits.
print("Categorical mask:", model.is_categorical_.tolist()) # Inspect recognized feature types.
print("Validation accuracy:", round(model.score(Xv, yv), 3)) # Evaluate unseen rows.
assert model.is_categorical_.tolist() == [True, False] # The nullable numeric column stays numeric.

### Reading the result
The boolean mask identifies city as categorical and value as numerical. This constructed relationship is deliberately easy. Compare native handling and a one-hot pipeline on representative development data with equivalent budgets before deciding which is better.

### Transfer exercise
Add a prediction record with an unseen city category and a missing numeric value. Compare its behavior with your one-hot pipeline.

## Regression prediction intervals with quantile loss
A point prediction describes one location of the target distribution. Quantile models estimate conditional lower and upper quantiles. An interval from the 0.1 and 0.9 quantiles has a nominal central coverage of 80%, but fitted models do not automatically achieve that coverage. Evaluate both coverage and interval width on held-out development data and inspect subgroups. Quantile crossing can occur when independently fitted lower and upper models disagree. A bootstrap of fitted mean predictions alone measures model-fit variability, not the full distribution of future outcomes: observation noise is missing.

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(42)
X = rng.uniform(0, 5, size=(300, 1)) # A numerical predictor.
y = 2 * X[:, 0] + rng.normal(size=300) * (0.5 + X[:, 0]) # Noise grows with the predictor.
Xt, Xv, yt, yv = train_test_split(X, y, random_state=42) # Validate on unseen development examples.
lower_model = GradientBoostingRegressor(loss="quantile", alpha=0.1, n_estimators=50, max_depth=2, random_state=42).fit(Xt, yt) # Estimate a lower conditional quantile.
upper_model = GradientBoostingRegressor(loss="quantile", alpha=0.9, n_estimators=50, max_depth=2, random_state=42).fit(Xt, yt) # Estimate an upper conditional quantile.
lo, hi = lower_model.predict(Xv), upper_model.predict(Xv) # Bounds for each validation row.
print("Empirical coverage:", round(np.mean((yv >= lo) & (yv <= hi)), 3)) # Fraction of outcomes within their intervals.
print("Mean signed width:", round(np.mean(hi - lo), 3)) # Width in target units; inspect crossings separately.
print("Crossed bounds:", int((lo > hi).sum())) # Do not hide invalid intervals.
assert lo.shape == hi.shape == yv.shape # One bound pair per example.

### Reading the result
Coverage describes these held-out outcomes. It can differ from 0.8 through estimation error, limited data, and distribution mismatch. Wide intervals can cover many outcomes without being useful. A narrow confidence interval for average MAE is a different object from a prediction interval for one future target.

### Transfer exercise
Compare nominal 80% and 90% quantile intervals using held-out coverage, width, and crossings. Report the noise assumptions and target units.

## Your turn
Compare two learning rates with a suitable iteration budget. Explain why the final test set cannot be used as the early-stopping set.

Try the task before opening the solution on the course website.